In [6]:
# Install a compatible PyTorch
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install PyG and its dependencies
!pip install torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

Looking in indexes: https://download.pytorch.org/whl/cu118
Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu118.html


In [29]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import torch
from torch_geometric.data import Data

# -----------------------------
# 1. Generate simulation data
# -----------------------------
NUM_FLIGHTS = 100
NUM_GATES = 25
AIRLINES = ['MU', 'CA', 'CZ', 'HU', 'ZH', 'HO']
TERMINALS = ['T1', 'T2']
AIRCRAFT_TYPES = ['A320', 'A330', 'B737', 'B777']
DELAY_PROB = 0.5
BUFFER_MIN = 20
np.random.seed(42)

# Generate gate info
gate_list = []
for i in range(NUM_GATES):
    gate_id = f"G{i+1:02d}"
    is_bridge = np.random.choice([1, 0], p=[0.7, 0.3])
    gate = {
        "Gate_ID": gate_id,
        "Terminal": random.choice(TERMINALS),
        "Is_Bridge": is_bridge,
        "Max_Aircraft": random.choice(AIRCRAFT_TYPES)
    }
    gate_list.append(gate)
gates_df = pd.DataFrame(gate_list)

# Generate remote stand mapping
remote_stands = [f"R{20+i:02d}" for i in range(NUM_GATES)]
gates_df["Stand_ID"] = np.where(
    gates_df["Is_Bridge"] == 1,
    gates_df["Gate_ID"],
    remote_stands[:len(gates_df)]
)

# Generate flight info
flight_list = []
base_time = datetime(2025, 8, 21, 6, 0, 0)
for _ in range(NUM_FLIGHTS):
    arr_time = base_time + timedelta(minutes=np.random.randint(0, 720))
    dep_time = arr_time + timedelta(minutes=np.random.randint(40, 90))
    airline = random.choice(AIRLINES)
    flight = {
        "Flight_ID": f"{airline}{np.random.randint(1000, 9999)}",
        "Aircraft_Type": random.choice(AIRCRAFT_TYPES),
        "Scheduled_ARR": arr_time,
        "Scheduled_DEP": dep_time,
        "Airline": airline,
        "Terminal": random.choice(TERMINALS),
        "Is_International": np.random.choice([0, 1], p=[0.8, 0.2]),
        "Delay_MIN": np.random.randint(10, 120) if np.random.rand() < DELAY_PROB else 0,
    }
    flight_list.append(flight)
flights_df = pd.DataFrame(flight_list)

# Compute actual arrival time
flights_df["Actual_ARR"] = flights_df["Scheduled_ARR"] + pd.to_timedelta(flights_df["Delay_MIN"], unit='m')
flights_df = flights_df.sort_values("Actual_ARR").reset_index(drop=True)

# ---------------------------------------------
# 2. Build Tripartite Graph: Flight → Stand → Gate
# ---------------------------------------------

# Node features for flights
flight_feats = []
for _, row in flights_df.iterrows():
    arr_min = row["Actual_ARR"].hour * 60 + row["Actual_ARR"].minute
    dep_min = row["Scheduled_DEP"].hour * 60 + row["Scheduled_DEP"].minute
    feat = [
        arr_min / 1440,
        dep_min / 1440,
        row["Is_International"],
        int(row["Aircraft_Type"] in ["A330", "B777"])
    ]
    flight_feats.append(feat)
flight_feats = torch.tensor(flight_feats, dtype=torch.float)

# Node features for stands and gates (simplified)
stand_feats = torch.rand(len(gates_df), 4)  # dummy features
gate_feats = torch.rand(len(gates_df), 4)   # dummy features

# Create edge connections
flight_to_stand_edge_index = torch.tensor([
    [i, j] for i in range(len(flights_df)) for j in range(len(gates_df))
], dtype=torch.long).t().contiguous()

stand_to_gate_edge_index = torch.tensor([
    [i, i] for i in range(len(gates_df))
], dtype=torch.long).t().contiguous()

# Combine all nodes and edges
x = torch.cat([flight_feats, stand_feats, gate_feats], dim=0)
flight_offset = 0
stand_offset = len(flight_feats)
gate_offset = stand_offset + len(stand_feats)

edge_index = torch.cat([
    flight_to_stand_edge_index + torch.tensor([[flight_offset], [stand_offset]]),
    stand_to_gate_edge_index + torch.tensor([[stand_offset], [gate_offset]])
], dim=1)

# Construct PyG Data object
data = Data(x=x, edge_index=edge_index)

# Output basic info
print(data)
print("Flight node features:", flight_feats.shape)
print("Stand node features:", stand_feats.shape)
print("Gate node features:", gate_feats.shape)


Data(x=[150, 4], edge_index=[2, 2525])
Flight node features: torch.Size([100, 4])
Stand node features: torch.Size([25, 4])
Gate node features: torch.Size([25, 4])


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GateAssignmentGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_gates):
        super().__init__()
        self.gcn1 = GCNConv(in_channels, hidden_channels)
        self.gcn2 = GCNConv(hidden_channels, hidden_channels)
        self.fc = nn.Linear(hidden_channels, num_gates)  # Predict gate logits

    def forward(self, x, edge_index, flight_mask):
        x = self.gcn1(x, edge_index)
        x = F.relu(x)
        x = self.gcn2(x, edge_index)
        x = F.relu(x)
        x = self.fc(x)
        return x[flight_mask]  # Only return logits for flight nodes


In [31]:
# Initialize allocation results
assigned_gates = []
assigned_stands = []
stand_usage = {}

for idx, row in flights_df.iterrows():
    arr = row["Scheduled_ARR"] - timedelta(minutes=BUFFER_MIN)
    dep = row["Scheduled_DEP"] + timedelta(minutes=BUFFER_MIN)
    assigned = False

    for _, gate_row in gates_df.iterrows():
        gate_id = gate_row["Gate_ID"]
        is_bridge = gate_row["Is_Bridge"]
        stand_id = gate_id if is_bridge else remote_gate_map[gate_id]

        usage = stand_usage.get(stand_id, [])
        conflict = any((arr < end and dep > start) for start, end in usage)

        if not conflict:
            assigned_gates.append(gate_id)
            assigned_stands.append(stand_id)
            stand_usage.setdefault(stand_id, []).append((arr, dep))
            assigned = True
            break

    if not assigned:
        assigned_gates.append(None)
        assigned_stands.append(None)

# Add to flights_df
flights_df["Assigned_Gate"] = assigned_gates
flights_df["Assigned_Stand"] = assigned_stands


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [32]:
# Create mapping from gate ID to gate index
gate_id_to_index = {gate_id: i for i, gate_id in enumerate(gates_df["Gate_ID"])}

# Find valid flights with assigned gates
valid_flight_indices = flights_df[flights_df["Assigned_Gate"].notnull()].index.tolist()

# Create label tensor (gate index for each valid flight)
labels = torch.tensor([
    gate_id_to_index[flights_df.loc[i, "Assigned_Gate"]] for i in valid_flight_indices
], dtype=torch.long)


In [33]:
# Create flight_mask (bool mask for flight nodes)
num_flights = len(flights_df)
flight_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
flight_mask[:num_flights] = True

# Instantiate model
model = GateAssignmentGNN(in_channels=4, hidden_channels=32, num_gates=len(gates_df))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()


# Training loop
model.train()
for epoch in range(100):
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, flight_mask)
    out = out[valid_flight_indices]  # Only use valid flights with labels
    loss = criterion(out, labels)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        pred = out.argmax(dim=1)
        acc = (pred == labels).float().mean().item()
        print(f"Epoch {epoch+1:03d} | Loss: {loss.item():.4f} | Acc: {acc*100:.2f}%")


Epoch 010 | Loss: 3.0160 | Acc: 10.00%
Epoch 020 | Loss: 2.8383 | Acc: 9.00%
Epoch 030 | Loss: 2.7498 | Acc: 13.00%
Epoch 040 | Loss: 2.6920 | Acc: 13.00%
Epoch 050 | Loss: 2.6400 | Acc: 13.00%
Epoch 060 | Loss: 2.5823 | Acc: 15.00%
Epoch 070 | Loss: 2.5165 | Acc: 15.00%
Epoch 080 | Loss: 2.4459 | Acc: 16.00%
Epoch 090 | Loss: 2.3839 | Acc: 16.00%
Epoch 100 | Loss: 2.3279 | Acc: 18.00%


In [34]:
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index, flight_mask)
    predicted_classes = logits.argmax(dim=1).cpu().numpy()

# Add prediction results to the DataFrame
flights_df["Predicted_Gate"] = [list(gate_id_to_class.keys())[i] for i in predicted_classes]
print(flights_df[["Flight_ID", "Assigned_Gate", "Predicted_Gate"]].head(10))


  Flight_ID Assigned_Gate Predicted_Gate
0    HU4157           G01            G01
1    CA4420           G02            G08
2    HO5380           G03            G03
3    ZH3491           G04            G04
4    HO7316           G05            G04
5    ZH1728           G06            G04
6    ZH5491           G07            G04
7    HO9392           G08            G03
8    HU2571           G09            G06
9    CA8390           G10            G06


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [35]:
# Select and rename relevant columns for export
export_df = flights_df[[
    "Flight_ID", "Aircraft_Type", "Airline", "Is_International",
    "Scheduled_ARR", "Scheduled_DEP", "Delay_MIN", "Actual_ARR",
    "Assigned_Gate", "Assigned_Stand"
]].copy()

# Sort by Scheduled_ARR for easier viewing
export_df.sort_values("Scheduled_ARR", inplace=True)

# Save to CSV
export_df.to_csv("simulated_GNN_flight_gate_assignment.csv", index=False, encoding="utf-8-sig")

print("Exported to simulated_GNN_flight_gate_assignment.csv")


Exported to simulated_GNN_flight_gate_assignment.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [22]:
from collections import Counter

usage_counter = Counter(assigned_stands)
for stand, count in sorted(usage_counter.items()):
    print(f"{stand}: used {count} times")

unassigned = sum(pd.isna(flights_df["Assigned_Stand"]))
print(f"\nUnassigned flights: {unassigned}")


G01: used 8 times
G04: used 6 times
G05: used 5 times
G06: used 6 times
G07: used 5 times
G09: used 4 times
G11: used 5 times
G14: used 4 times
G15: used 3 times
G16: used 4 times
G17: used 3 times
G18: used 3 times
G19: used 3 times
G20: used 2 times
G21: used 2 times
G22: used 3 times
G23: used 2 times
G24: used 1 times
G25: used 1 times
R20: used 6 times
R21: used 5 times
R22: used 6 times
R23: used 4 times
R24: used 5 times
R25: used 4 times

Unassigned flights: 0


In [23]:
!pip install gymnasium
!pip install stable-baselines3[extra]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 4.9 MB/s eta 0:00:00


In [24]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
from datetime import timedelta


class RealTimeGateEnv(gym.Env):
    """
    Environment for real-time gate reallocation based on delays.
    Each step processes a flight; the agent decides to reassign it or not.
    """
    def __init__(self, flights_df, gates_df, buffer_min=20):
        super().__init__()
        self.flights_df = flights_df.copy()
        self.gates_df = gates_df.copy()
        self.buffer_min = buffer_min
        self.num_gates = len(gates_df)
        self.num_flights = len(flights_df)

        self.action_space = spaces.Discrete(self.num_gates + 1)  # 0~n-1 = reassign to gate; n = keep original
        self.observation_space = spaces.Dict({
            "flight_features": spaces.Box(low=0, high=1, shape=(6,), dtype=np.float32),
            "gate_mask": spaces.MultiBinary(self.num_gates)
        })

        self.remote_gate_map = self._build_remote_gate_map()
        self.current_idx = 0
        self.stand_usage = {g: [] for g in gates_df["Gate_ID"]}

    def _build_remote_gate_map(self):
        # Map non-bridge gates to remote stand IDs
        remote_stands = [f"R{20+i:02d}" for i in range(self.num_gates)]
        return dict(zip(self.gates_df[self.gates_df["Is_Bridge"] == 0]["Gate_ID"], remote_stands))

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.current_idx = 0
        self.stand_usage = {g: [] for g in self.gates_df["Gate_ID"]}
        return self._get_obs(), {}

    def _get_obs(self):
        if self.current_idx >= self.num_flights:
            return {
                "flight_features": np.zeros(6, dtype=np.float32),
                "gate_mask": np.zeros(self.num_gates, dtype=np.int8)
            }

        row = self.flights_df.iloc[self.current_idx]
        arr = row["Actual_ARR"] - timedelta(minutes=self.buffer_min)
        dep = row["Scheduled_DEP"] + timedelta(minutes=self.buffer_min)

        gate_mask = []
        for _, gate_row in self.gates_df.iterrows():
            gate_id = gate_row["Gate_ID"]
            usage = self.stand_usage[gate_id]
            conflict = any((arr < end and dep > start) for start, end in usage)
            gate_mask.append(0 if conflict else 1)

        # Features: arrival, dep, delay, intl, wide-body, original_gate_id
        arr_min = row["Actual_ARR"].hour * 60 + row["Actual_ARR"].minute
        dep_min = row["Scheduled_DEP"].hour * 60 + row["Scheduled_DEP"].minute
        features = np.array([
            arr_min / 1440,
            dep_min / 1440,
            row["Delay_MIN"] / 180,
            row["Is_International"],
            int(row["Aircraft_Type"] in ["A330", "B777"]),
            0.0 if pd.isna(row["Assigned_Gate"]) else int(row["Assigned_Gate"][1:]) / self.num_gates
        ], dtype=np.float32)

        return {
            "flight_features": features,
            "gate_mask": np.array(gate_mask, dtype=np.int8)
        }

    def step(self, action):
        info = {}
        if self.current_idx >= self.num_flights:
            return self._get_obs(), 0.0, True, True, info

        row = self.flights_df.iloc[self.current_idx]
        arr = row["Actual_ARR"] - timedelta(minutes=self.buffer_min)
        dep = row["Scheduled_DEP"] + timedelta(minutes=self.buffer_min)

        assigned_gate = row["Assigned_Gate"]
        gate_id = None
        reward = 0

        if action == self.num_gates:
            # Keep original gate
            gate_id = assigned_gate
        else:
            gate_id = self.gates_df.iloc[action]["Gate_ID"]

        # Check conflict
        usage = self.stand_usage.get(gate_id, [])
        conflict = any((arr < end and dep > start) for start, end in usage)

        if conflict or pd.isna(gate_id):
            reward = -5
        else:
            reward = 1 if gate_id == assigned_gate else 0.5  # encourage reuse
            self.stand_usage[gate_id].append((arr, dep))

        self.current_idx += 1
        terminated = self.current_idx >= self.num_flights
        truncated = False
        return self._get_obs(), reward, terminated, truncated, info

    def render(self):
        print(f"Flight {self.current_idx}")


In [25]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# Create and wrap environment
env = RealTimeGateEnv(flights_df, gates_df)
vec_env = DummyVecEnv([lambda: env])

# Train agent
model = PPO("MultiInputPolicy", vec_env, verbose=1)
model.learn(total_timesteps=100000)

# Test agent
# Test agent & collect reassignment results
obs, _ = env.reset()
total_reward = 0
reassigned_flights = []

while True:
    action, _ = model.predict(obs)
    obs, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward

    # Collect reallocation info
    if env.current_idx <= env.num_flights:
        flight_row = env.flights_df.iloc[env.current_idx - 1]  # current flight
        assigned_gate = flight_row["Assigned_Gate"] if action == env.num_gates else env.gates_df.iloc[action]["Gate_ID"]
        stand_id = assigned_gate if assigned_gate in env.stand_usage else env.remote_gate_map.get(assigned_gate, None)

        reassigned_flights.append({
            "Flight_ID": flight_row["Flight_ID"],
            "Assigned_Gate_Reassigned": assigned_gate,
            "Assigned_Stand_Reassigned": stand_id,
            "Reward": reward  # Add reward here
        })

    if terminated or truncated:
        break

print(f"✅ Total reward from real-time reallocation: {total_reward}")

# Convert collected info to reassigned_df
reassigned_df = pd.DataFrame(reassigned_flights)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cpu device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------
| time/              |      |
|    fps             | 346  |
|    iterations      | 1    |
|    time_elapsed    | 5    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 291         |
|    iterations           | 2           |
|    time_elapsed         | 14          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008084816 |
|    clip_fraction        | 0.0534      |
|    clip_range           | 0.2         |
|    entropy_loss         | -3.25       |
|    explained_variance   | -0.00479    |
|    learning_rate        | 0.0003      |
|    loss                 | 28.7        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0202     |
|    value_loss           | 134         |
-----------------------------------------
----------------------------------

In [26]:
import pandas as pd

# 1. Merge reassignment results and rewards into the original flights dataframe
full_df = flights_df.copy()
full_df = full_df.merge(
    reassigned_df[['Flight_ID', 'Assigned_Gate_Reassigned', 'Assigned_Stand_Reassigned', 'Reward']],
    on='Flight_ID',
    how='left'
)

# 2. Check whether reassignment occurred (either gate or stand changed)
def was_reassigned(row):
    return int(
        row['Assigned_Gate'] != row['Assigned_Gate_Reassigned'] or
        row['Assigned_Stand'] != row['Assigned_Stand_Reassigned']
    )

full_df['Reassigned'] = full_df.apply(was_reassigned, axis=1)

# 3. Summary statistics
total_flights = len(full_df)
total_reassigned = full_df['Reassigned'].sum()
reassignment_rate = total_reassigned / total_flights * 100 if total_flights else 0

print(f"Total Flights: {total_flights}")
print(f"Reassigned Flights: {total_reassigned}")
print(f"Reassignment Rate (All Flights): {reassignment_rate:.2f}%")

# 4. Preview a sample of the results
print("\nSample of all flights with reassignment and reward info:")
print(full_df[['Flight_ID', 'Assigned_Gate', 'Assigned_Gate_Reassigned', 'Reassigned', 'Reward']].head())

# 5. Export full results to CSV
full_df.to_csv("all_flight_reassignment_analysis.csv", index=False)
print("✅ CSV file saved as 'all_flight_reassignment_analysis.csv'")


Total Flights: 100
Reassigned Flights: 93
Reassignment Rate (All Flights): 93.00%

Sample of all flights with reassignment and reward info:
  Flight_ID Assigned_Gate Assigned_Gate_Reassigned  Reassigned  Reward
0    CZ4157           G01                      G07           1     0.5
1    ZH4420           G02                      G11           1     0.5
2    CZ5380           G03                      G01           1     0.5
3    CZ3491           G04                      G18           1     0.5
4    MU7316           G05                      G22           1     0.5
✅ CSV file saved as 'all_flight_reassignment_analysis.csv'


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
